# **Default 5G Deployment Example**

In this example, we will deploy a 5G network using the default 5G parameters of the `ifran` orchestrator, which are:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OAI gNB
- A simulated OAI NR UE

First, we install the required dependencies:

## Install Dependencies:

Install the `ifran` library:

In [ ]:
! pip install ifran

Install the Ansible dependencies:

In [ ]:
! ansible-galaxy install -r ../../ansible/requirements.yaml

Install the SLICES CLI used for the BI to create VMs, and for POS to provision baremetal serevers:

In [ ]:
! pip install slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Install the SLICES POS CLI:

In [ ]:
! pip install slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Authenticate with your SLICES account:

In [ ]:
! slices auth login

Make sure you have an SSH key that you add to SLICES using the command:

```bash
slices pubkey register /path/to/ssh/key
```

which will be use to grant SSH access to the provisioned SLICES machines.

## Input 5G Network Description:

The `ifran` orchestrator keeps the logical network description separate from the deployment-specific choices and evolving deployment state, so the same experiment can be adapted to different implementations or testbeds.

To achieve that, it uses two separate models:
- The metamodel describing the entire network and its relationships using in pure 3GPP-standard way
- The deployment model describing implementation choices, target nodes, provider settings, and operational state

To deploy, you invoke lifecycle managers for the work you need. For example, the `ProvisioningManagement` class reserves or creates resources, `KubernetesManagement` prepares nodes and forms a cluster, `CoreManagement`, `RANManagement`, `RICManagement`, and `UEManagement` prepare and install workloads on their respective 5G components.

These stages are composable management operations, not necessarily one automatic "deploy everything" command.

Each stage exposes its particular set of actions that run the corresponding Ansible roles and Helm actions, often concurrently where possible.

To get a Meta Model, we can initiate an empty `MetaModel` object that will use a default set of parameters to provide a 3GPP-standard 5G network description:

In [1]:
from ifran import MetaModel

metamodel = MetaModel()

To see the full content of this default Meta Model:

In [2]:
print(metamodel.to_yaml(exclude_defaults=False))

core:
  components:
    amf:
      category: core
      type: amf
      config:
        sbi:
          interfaceName: eth0
          ipv4EndPointAddress: 0.0.0.0
          ipv6EndPointAddress: null
          port: 80
          protocol: TCP
          scheme: http
          apiVersion: v1
          tls:
            enabled: false
            certificate: null
            privateKey: null
        nrfUri: null
        n2:
          interfaceName: eth0
          ipv4EndPointAddress: 0.0.0.0
          ipv6EndPointAddress: null
          port: 38412
          protocol: SCTP
        guamiList:
        - plmnId:
            mcc: '001'
            mnc: '01'
            mncLength: 2
            plmnId: '00101'
          amfRegionId: '02'
          amfSetId: '001'
          amfPointer: '00'
          amfId: '020040'
        taiList:
        - plmnId:
            mcc: '001'
            mnc: '01'
            mncLength: 2
            plmnId: '00101'
          tac: 1
        plmnList:
        - plmnI

By default, the metamodel contains:

- Open5GS core with: AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR
- OAI gNB
- A simulated OAI NR UE

Next, to get the deployment model, we need initiate a `DeploymentModel` object.

The easiest way to do so is by providing a YAML input that describes our deployment.

By default, the `DeployemntModel` defines 4 large-size VMs on the SLICES BI `fr-sophia2-bi-vm1` infrastructure.

However, we will have to modify some parameters to ensure the VM provisioning is configured correctly.

In [3]:
from ifran import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
params:
  provisioning:
    project: post5g-beta
    experiment: ifran
    create_experiment: true
    extra_path: /Users/zimabrou/Projects/5g_blueprint/venv/bin
"""
)

Here, we specified:
- The absolute path to where the SLICES CLI was installed on the Python venv
- The SLICES project to use: `post5g-beta`
- The SLICES experiment: `ifran`

Make sure to create the SLICES experiment beforehand using:

```bash
slices project use post5g-beta
slices experiment create 5gbp-exp --duration 1d
```

or add `create_experiment: true` to your deployment model `params.provisioning` to create the experiment automatically.

By default, the created experiment duration is 1 day, and the VM duration is 3 hours. To change that, use the `experiment_duration` and `node_duration` parameters respectively.

> *Note*: Check if your Ansible roles path was found automatically:

In [4]:
deployment_model.params.ansible.roles_paths

['/Users/zimabrou/Projects/5g_blueprint/venv2/lib/python3.14/site-packages/ansible/roles']

> *Note*: If the list is empty, your Ansible roles path wan not found. If so, before continuing, please also add the absolute path to your Ansible roles in the `params.ansible.roles_paths` list.

## Provisioning:

To provision the deployment VMs, we will invoke the `ProvisioningManagement` class from the `ifran` library and initiate it with the `metamodel` and `deployment_model`:

In [5]:
from ifran import ProvisioningManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)

> *Note*: During the initialization phase of each deployment stage management class, a reconcilation phase ensures the deployment model is  synchronized with the meta model, which takes precedence. 

To provision the nodes, we run the `provision()` action provided by this deployment management class, which will internally run the corresponding `provisioning/bi/provision` Ansible role:

In [6]:
await prov.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ******************************

> *Note*: All the deployment actions are asynchronous function that have to be awaited.

Once the provisioning is finished, the deployment model will be updated to contain the SSH informations about the provisoned nodes needed to generate an Ansible inventory used for the subsequent deployement stages.

In [7]:
print(prov.get_ansible_inventory())

[core_nodes]
node1 ansible_host=10.123.203.185 ansible_port=22 ansible_user=ziyad ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.theblueprintfactory.org -p 22"'
node2 ansible_host=10.123.164.37 ansible_port=22 ansible_user=ziyad ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ControlMaster=auto -o ControlPersist=7200s -o ControlPath=/tmp/cm-%C -o ProxyCommand="ssh -o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -W %h:%p proxy@bastion1.theblueprintfactory.org -p 22"'
node3 ansible_host=10.123.32.109 ansible_port=22 ansible_user=ziyad ansible_ssh_common_args='-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null -o Serv

You can run commands on the provisioned nodes using the the `run_node_command` function. For example:

In [8]:
print((await prov.run_node_command(command=["apt", "update"], nodes=["node1"], become=True, quiet=True)))

Hit:1 http://fr.archive.ubuntu.com/ubuntu noble InRelease
Get:2 http://fr.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:3 http://fr.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:4 http://fr.archive.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 http://fr.archive.ubuntu.com/ubuntu noble/universe amd64 Packages [15.0 MB]
Get:6 http://fr.archive.ubuntu.com/ubuntu noble/universe Translation-en [5982 kB]
Get:7 http://fr.archive.ubuntu.com/ubuntu noble/universe amd64 Components [3871 kB]
Get:8 http://fr.archive.ubuntu.com/ubuntu noble/universe amd64 c-n-f Metadata [301 kB]
Get:9 http://fr.archive.ubuntu.com/ubuntu noble/multiverse amd64 Packages [269 kB]
Get:10 http://fr.archive.ubuntu.com/ubuntu noble/multiverse Translation-en [118 kB]
Get:11 http://fr.archive.ubuntu.com/ubuntu noble/multiverse amd64 Components [35.0 kB]
Get:12 http://fr.archive.ubuntu.com/ubuntu noble/multiverse amd64 c-n-f Metadata [8328 B]
Get:13 http://fr.archive.ubuntu.com

> *Note*: The `run_ue_command` can abe used to run commands on provisioned COTS UEs that allow SSH access.

## Kubernetes Deployment:

Next, to deploy Kuberenetes on the provisioned nodes, we will invoke the `KubernetesManagement` class.

To initiate it, we can either use the `output()` function of the previous deployment stage to get the metamodel (which is unchanged) and the deployment model (which now contains the SSH info about the nodes) output, and then use them as input just like we did before for the `ProvisioningManagement`.

Or more conveniantly, we can give the entire previous deployment class as input for the next one like this:

In [9]:
from ifran import K8SManagement

k8s = K8SManagement(preceding=prov)

To deploy Kuberenetes, we will call the `deploy()` action fucntion of the `KubernetesManagement` class, which is a wrapper fucntion that internally calls the `prepare()` actions to install the necessary dependencies on all the provisioned nodes, then the `create()` action to initiate the Kubernetes cluster on the master node (by default the first node in the `nodes` section of the deployment model) and get the Kuberenetes join command, and finally the `join()` action to add the remaining worker nodes on the newly created cluster.

Again, these actions execute internally the corresponding `k8s/<action>` ansible role.

In [10]:
await k8s.deploy()


PLAY [node3] *******************************************************************

PLAY [node2] *******************************************************************

PLAY [node1] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node3'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node2'] ****************

PLAY [node4] *******************************************************************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node1'] ****************

TASK [k8s/prepare : Set time zone to 'Europe/Paris' on 'node4'] ****************
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [k8

Once the Kubernetes cluster is created we can move to the actual 5G deployment.

## 5G Core Deployment:

To deploy the 5G core network, we again invoke the corresponding `CoreManagement` class and initiate it using its preceding class: 

In [11]:
from ifran import CoreManagement

core = CoreManagement(preceding=k8s)

> *Note*: The 5G deployment management classes (`CoreManagement`, `RANMamanegemt`, `RICManagement` and `UEManagement`) all have a reconcilation phase that handles resolving the cross-references between the different 5G components (e.g. gNB referencing the AMF) and populating the container images and other Kuberenetes parameters that can also be changed manually through the deployment model.

Again, we call the `deploy()` action fucntion, which internally will prepare the core network components (AMF, AUSF, BSF, NSSF, NRF, PCF, SMF, UPF, UDM, UDR) and the database by:
- copying the Helm chart file to the master node
- rendering the Jinja templates from the metamodel and deployment model configuration parameters
- installing the Helm chart and getting the installed pod network information
- registering the subscribers in the `ues` section of the metamodel on the 5G core database

In [12]:
await core.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-pcf' chart] ******

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-smf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-udr' chart] ******

PLAY [node1] *******************************************************************

PLAY [node1] *******************************************************************

TASK [helm/pr

## RAN Deployment:

Once the 5G core has beed deployed, the deployment model should now contain the necessary network information about the AMF for the gNB to connect to it.

To deploy the OAI RFSIM RAN, we again call the corresponding `RANManagement` class and initiate it using the preceding class:

In [13]:
from ifran import RANManagement

ran = RANManagement(preceding=core)

Then call the `deploy()` action function to prepare and install the gNB Helm chart:

In [14]:
await ran.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-gnb' chart] **********
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-gnb' if provided] ***
skipping: [node1]

TASK [helm/prepare : Copy local built-in 'oai-gnb' Helm chart directory for 'oai-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-gnb'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'oai-gnb' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm files for 'oai-gnb'] ***

For all the available 5G deployement management classes, we can use the `get_logs()` action function that will fetch logs from the deployed 5G components. A single component returns text; multiple components return a dictionary keyed by component name.

> *Note*: By default, the `ifran` orchestrator fetches the logs from the deployed 5G component since the last 1 hour, but again, these paramaters can be changed for each component in the deployement model.

In [15]:
print(await ran.get_logs(components=["gnb"], quiet=True))

/proc/sys/kernel/core_pattern=|/usr/share/apport/apport -p%p -s%s -c%c -d%d -P%P -u%u -g%g -F%F -- %E
No configuration file /opt/oai-gnb/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

Active_gNBs:
  - gnb
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  ##### Identification parameters:
  - gNB_ID: 0x1

    gNB_name: gnb

    tracking_area_code: 1

    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1

    ##### Physical parameters:
    min_rxtxtime: 6

    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        absoluteFrequencySSB: 621312
        dl_absoluteFrequencyPointA: 620040
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120
        dl_subcarrierSpacing: 1
        dl_carrierBandwidth: 1

> *Note*: The `quiet=True` option is used to supress the Ansible output to stdout.

As we can see, the gNB has been deployed successfully. We can now deloy the UE.

## UE Deployement:

To deploy the OAI RFSIM NR UE defined in the meta and deployemnt models, we again initiate the corresponding `UEManagement` deployement management class from the previous one to maintain the deployment state so far, which includes the necessary gNB pod network information that will be used to configure and deploy the simulated UE correctly:

In [16]:
from ifran import UEManagement

ue = UEManagement(preceding=ran)

To deploy, we again call the `deploy()` action function:

In [17]:
await ue.deploy()


PLAY [node1] *******************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue' chart] *******
[WARNING]: Host 'node1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node1]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-sim-ue' if provided] ***
skipping: [node1]

TASK [helm/prepare : Copy local built-in 'oai-containerized' Helm chart directory for 'oai-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render built-in Helm Chart.yaml templated file for 'oai-sim-ue'] ***
changed: [node1]

TASK [helm/prepare : Render custom templated Helm files for 'oai-sim-ue' if provided] ***
skipping: [node1]

TASK [helm/prepare : Render built-in templated Helm fi

Once deployed, we can again use the `get_logs()` function to check the logs of our UE deployment whitout having to manually access the Kuberenetes pod:

In [18]:
print(await ue.get_logs(components=["sim_ue"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

As we can see, the UE seems to have connected successfully.

To SSH into the SLICES BI VM and check your deployement, you can run:

```bash
slices bi --infra fr-sophia2-bi-vm1 ssh node1
```

> Make sure you use the same SLICES experiment name used in the deployment model.

From there, you can do:

```bash
sudo su -
```

to switch to the priviledged user, and use the standard `kubectl` commands to check your deployment.

In this case, the command to access the UE pod will be:

```bash
kubectl exec deployment/oai-sim-ue -n oai-ue -it -- bash
```

> *Note*: Each 5G component defined in the meta and deployement model follows these naming conventions:
> - namespace: `<implementation>-<category>` (e.g. AMF is in the `core` category and is deployed using the `Open5GS` implemetation, so it will be in the `open5gs-core` namespace)
> - chart name: `<implementation>-<name>` (e.g. we deployed an `OAI` gNB called `gnb`, so its deployement name will be `oai-gnb` in the namespace `oai-ran`)

A more convenient option is to use the `run_command` from the deployment management class to execute a command directly on the pod. For example:

In [19]:
print((await ue.run_command(command=["ip", "addr"], quiet=True)))

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0@if11: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9166 qdisc noqueue state UP group default qlen 1000
    link/ether 4e:25:49:a4:33:9e brd ff:ff:ff:ff:ff:ff link-netnsid 0
    inet 10.244.135.6/32 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::4c25:49ff:fea4:339e/64 scope link 
       valid_lft forever preferred_lft forever
3: oaitun_ue1: <POINTOPOINT,NOARP,UP,LOWER_UP> mtu 1500 qdisc fq_codel state UNKNOWN group default qlen 500
    link/none 
    inet 11.1.1.2/24 scope global oaitun_ue1
       valid_lft forever preferred_lft forever
    inet6 fe80::6520:c8ac:729e:6e84/64 scope link stable-privacy 
       valid_lft forever preferred_lft forever



## Adding a new component on the fly:

To add a new 5G component to the metamodel, we can use the `add_component()` action funntion in the 5G deployemnt management classes to add a new corresponfing 5G component. For example, we can add a UE from the `UEManagement` class we have.

For that, we can use the `UE` meta model to specify a `supi` field different from the default value of `001010000000100`:

In [20]:
from ifran import UE

ue.add_component(name="sim_ue2", component_meta=UE.from_yaml(
"""
config:
    supi: '001010000000101'
"""
))

This action only adds the UE to the metamodel and calls the synchronize internally to update the deployment model.

To actually add the UE to the 5G network, we first have to register it in the database using the `register_ues()` action function in the `CoreManagement` class:

In [21]:
core = CoreManagement(preceding=ue)

In [22]:
await core.register_ues(ues=["sim_ue2"], quiet=True)

Next, we deploy the new `sim_ue2` using the `deploy` action in the `UEManagement` class again:

In [23]:
await ue.deploy(components=["sim_ue2"], quiet=True)

Once finished, we fetch and print the UE's logs to ensure the deployment was successfull:

In [24]:
print(await ue.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-f80a88326801f5417405dea0b174879bac8940e0 Ab

We can also test by pinging the first UE:

In [25]:
print((await ue.run_command(command=["ping", "11.1.1.2", "-c", "5", "-I", "oaitun_ue1"], components=["sim_ue2"], quiet=True)))

PING 11.1.1.2 (11.1.1.2) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
64 bytes from 11.1.1.2: icmp_seq=1 ttl=64 time=64.9 ms
64 bytes from 11.1.1.2: icmp_seq=2 ttl=64 time=70.1 ms
64 bytes from 11.1.1.2: icmp_seq=3 ttl=64 time=69.4 ms
64 bytes from 11.1.1.2: icmp_seq=4 ttl=64 time=65.2 ms
64 bytes from 11.1.1.2: icmp_seq=5 ttl=64 time=61.4 ms

--- 11.1.1.2 ping statistics ---
5 packets transmitted, 5 received, 0% packet loss, time 4004ms
rtt min/avg/max/mdev = 61.350/66.185/70.113/3.211 ms



To retreive the metamodel and deployement model output of your deployement managememt classes, you can use the `output()` function:

In [26]:
ue.output(metamodel_file="output_metamodel.yaml", deployment_model_file="output_deployment_model.yaml")

(MetaModel(core=Core(components={'amf': AMFComponent(category=<Category5G.CORE: 'core'>, type=<CoreComponentType.AMF: 'amf'>, config=AMFConfig(sbi=SBI(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=80, protocol='TCP', scheme='http', apiVersion='v1', tls=TLS(enabled=False, certificate=None, privateKey=None)), nrfUri=None, n2=NGAP(interfaceName='eth0', ipv4EndPointAddress=IPv4Address('0.0.0.0'), ipv6EndPointAddress=None, port=38412, protocol='SCTP'), guamiList=[GUAMI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), amfRegionId=2, amfSetId=1, amfPointer=0, amfId=131136)], taiList=[TAI(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), tac=1)], plmnList=[PLMNSupportItem(plmnId=PLMN(mcc=1, mnc=1, mncLength=2, plmnId='00101'), snssaiList=['internet'])], securityAlgorithmConfiguration=SecurityAlgorithmConfiguration(integrityOrder=[<IntegrityAlgorithm.NIA2: 'nia2'>, <IntegrityAlgorithm.NIA1: 'nia1'>, <IntegrityAlgorithm.NIA0: 'nia0'

> *Note*: All your Ansible run logs are stored by action, component and execution timestamp in `~/ifran/ansible-runner` by default, which you can of course change using the `runner_path` parameter in the `params.ansible` section of the deployment model.

To destroy the VMs, we can again invoke the `ProvisioningManagement` class and call the `unprovision` action fucntion:

In [27]:
# await prov.unprovision()

> *Note*: most actions have an opposite action to undo what the original action has done, for example, we have the "remove" action to undo the "deploy" action, the "unprovision" action to undo the "provision" action ... etc.

## `MiscellaneousManagement` class:

The `MiscellaneousManagement` class is the `ifran` orchestrator is a flexible extension point.

It runs custom Ansible playbooks against the current deployment, supplying the merged metamodel and deployment model by default, as well as the default Ansible inventory files used by the built-in roles.

It can collect selected playbook outputs and store them in the deployment model's `miscellaneous` section, to allow experiment-specific tasks that don’t belong to the standard provisioning, Kubernetes, core, RAN, RIC, or UE stages.